# Praat Scripts (日本語版) - 音声区間の一括切り出し (Google Colab版) 🎙️

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okawawaka/praat-scripts-ja/blob/main/notebooks/extract_intervals_to_wav.ipynb)

本ノートブックは、`scripts/segmentation/extract_intervals_to_wav.praat` と**全く同一のロジック**で、TextGridの指定Tierのラベル区間に従って音声を個別WAVファイルに一括切り出しするツールです。

切り出された個別音声ファイルは、**自動的にZIPアーカイブにまとめられて一括ダウンロード**されます。

### Step 1: 必要なライブラリのインストール

In [ ]:
!pip install -q praat-parselmouth textgrid

### Step 2: 切り出し条件の設定

In [ ]:
# @title 【パラメータ設定】 { run: "auto" }
target_tier = 1 # @param {type:"integer"}
skip_empty_intervals = True # @param {type:"boolean"}
include_file_index = True # @param {type:"boolean"}

print(f"設定完了: Tier={target_tier}, 空白除外={skip_empty_intervals}, 連番付与={include_file_index}")

### Step 3: 音声ファイル(.wav)とTextGrid(.TextGrid)をアップロード
下のセルを実行して表示される「ファイル選択」ボタンから、**切り出したいペア（.wav と .TextGrid）をまとめて選択**してください。

In [ ]:
import os
import re
import zipfile
import tempfile
import parselmouth
import textgrid
from google.colab import files

print("▼ 下の『ファイル選択』ボタンをクリックして、.wav と .TextGrid ファイルをまとめて選択してください：")
uploaded = files.upload()

if not uploaded:
    print("ファイルが選択されませんでした。")
else:
    temp_dir = tempfile.mkdtemp()
    output_dir = os.path.join(temp_dir, "extracted_wavs")
    os.makedirs(output_dir, exist_ok=True)

    for fname, data in uploaded.items():
        with open(os.path.join(temp_dir, fname), "wb") as f:
            f.write(data)

    wav_files = [f for f in uploaded.keys() if f.lower().endswith(".wav")]
    total_cut = 0

    for w_name in wav_files:
        base = os.path.splitext(w_name)[0]
        tg_candidates = [base + ".TextGrid", base + ".textgrid"]
        tg_name = next((c for c in tg_candidates if c in uploaded), None)

        if not tg_name:
            print(f"スキップ: {w_name} に対応する TextGrid が見つかりません。")
            continue

        wav_path = os.path.join(temp_dir, w_name)
        tg_path = os.path.join(temp_dir, tg_name)

        try:
            sound = parselmouth.Sound(wav_path)
            tg = textgrid.TextGrid.fromFile(tg_path)

            if target_tier <= len(tg.tiers):
                tier = tg.tiers[target_tier - 1]
                if isinstance(tier, textgrid.IntervalTier):
                    file_cut = 0
                    for idx, interval in enumerate(tier, start=1):
                        label = interval.mark.strip()
                        if skip_empty_intervals and label == "":
                            continue

                        start_t = interval.minTime
                        end_t = interval.maxTime
                        duration = end_t - start_t

                        if duration > 0.001:
                            # Praatと同一のExtract part処理
                            part = sound.extract_part(from_time=start_t, to_time=end_t, preserve_times=False)
                            safe_label = re.sub(r'[\\/:*?"<>|]', '_', label)
                            if not safe_label:
                                safe_label = "interval"

                            if include_file_index:
                                out_fname = f"{base}_{idx}_{safe_label}.wav"
                            else:
                                out_fname = f"{base}_{safe_label}.wav"

                            part.save(os.path.join(output_dir, out_fname), "WAV")
                            total_cut += 1
                            file_cut += 1
                    print(f"{base}: {file_cut} 区間を切り出しました。")
        except Exception as e:
            print(f"エラー ({base}): {e}")

    print(f"\n切り出し完了: 合計 {total_cut} 個の音声ファイルを生成しました。")

### Step 4: 切り出し音声をZIPで一括ダウンロード
下のセルを実行すると、生成されたすべてのWAVファイルをまとめた `extracted_wavs.zip` がダウンロードされます。

In [ ]:
if 'total_cut' in locals() and total_cut > 0:
    zip_path = "extracted_wavs.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_in_dir in os.walk(output_dir):
            for f in files_in_dir:
                zipf.write(os.path.join(root, f), arcname=f)

    print(f"{zip_path} を作成しました。ダウンロードを開始します...")
    files.download(zip_path)
else:
    print("ダウンロード対象の音声ファイルがありません。")